### Random walk in 2D

In [ ]:
# Required imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# for animations and interactivity
from IPython.display import display, HTML
import ipywidgets as widgets

In [ ]:
# -----------------------------
# Core random walk function
# -----------------------------
def random_walk(nt, latt_type):
    """
    Random walk on a 2D lattice
    Inputs:
        nt [integer] = number of desired jumps (i.e., time steps)
        latt_type [string] = lattice geometry, current options
                             are 'square' and 'triangle'
    Outputs:
        rs2 [array] = square displacement at each time step
        x   [array] = x-coordinate at each time step
        y   [array] = y-coordinate at each time step
    """
    x = np.zeros(nt + 1)
    y = np.zeros(nt + 1)
    rs2 = np.zeros(nt + 1)

    x[0] = 0
    y[0] = 0
    rs2[0] = 0

    if latt_type == 'square':
        fd = np.floor(4 * np.random.rand(nt)).astype(int)
        delx = np.array([1, -1, 0, 0])
        dely = np.array([0, 0, 1, -1])
    elif latt_type == 'triangle':
        fd = np.floor(6 * np.random.rand(nt)).astype(int)
        delx = np.array([1, -1, 0.5, -0.5, 0.5, -0.5])
        dely = np.array([
            0,
            0,
            np.sqrt(3.) / 2.,
            np.sqrt(3.) / 2.,
            -np.sqrt(3.) / 2.,
            -np.sqrt(3.) / 2.
        ])
    else:
        raise ValueError("Lattice type not implemented! Use 'square' or 'triangle'.")

    for j in range(nt):
        step = fd[j]
        x[j + 1] = x[j] + delx[step]
        y[j + 1] = y[j] + dely[step]
        rs2[j + 1] = x[j + 1]**2 + y[j + 1]**2

    return rs2, x, y

# -----------------------------
# Plotting helpers
# -----------------------------
def plot_static(x, y, lattice, nsteps):
    fig, ax = plt.subplots()
    ax.set_title(f"Random walk trajectory on {lattice} lattice with {nsteps} steps")
    ax.set_xlabel('x positions')
    ax.set_ylabel('y positions')
    ax.plot(x, y)
    plt.show()


def plot_animated(x, y, lattice, nsteps, interval=50):
    fig, ax = plt.subplots()
    line, = ax.plot([], [])

    ax.set(
        xlim=(min(x), max(x)),
        ylim=(min(y), max(y)),
        xlabel='X',
        ylabel='Y',
        title=f"Random walk trajectory on {lattice} lattice with {nsteps} steps"
    )

    def update(frame_num):
        line.set_xdata(x[:frame_num])
        line.set_ydata(y[:frame_num])
        return line,

    ani = animation.FuncAnimation(fig, update, nsteps, interval=interval)
    html = HTML(ani.to_jshtml())
    plt.close(fig)
    display(html)


In [ ]:
# -----------------------------
# ipywidgets UI
# -----------------------------
lattice_widget = widgets.ToggleButtons(
    options=['square', 'triangle'],
    description='Lattice:',
    value='square'
)

num_steps_widget = widgets.IntSlider(
    value=100,
    min=0,
    max=1000,
    step=10,
    description='Steps:',
    continuous_update=False
)

display_type_widget = widgets.ToggleButtons(
    options=['static (fast)', 'animated (slow)'],
    description='Display:',
    value='static (fast)'
)

run_button = widgets.Button(
    description='Run simulation',
    button_style='primary'
)

output = widgets.Output()


def run_simulation(_):
    with output:
        output.clear_output()
        lattice = lattice_widget.value
        nsteps = int(num_steps_widget.value)
        disp_type = display_type_widget.value

        rs2, x, y = random_walk(nsteps, latt_type=lattice)

        if disp_type == 'static (fast)':
            plot_static(x, y, lattice, nsteps)
        else:
            plot_animated(x, y, lattice, nsteps)


run_button.on_click(run_simulation)

ui = widgets.VBox([
    widgets.HBox([lattice_widget, display_type_widget]),
    num_steps_widget,
    run_button
])

display(ui, output)